# 01 — Exploratory Data Analysis
**Dataset:** `credit_risk_dataset.csv`  
**Problem:** Binary classification — predict `loan_status` (0 = good, 1 = default)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

DATA_PATH = os.path.join('..', 'data', 'raw', 'credit_risk_dataset.csv')
df = pd.read_csv(DATA_PATH)
print('Loaded:', df.shape)

## 1. Basic Info — Shape, Dtypes, Describe

In [ ]:
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nStatistical summary:')
display(df.describe(include='all').T)

## 2. Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)
print('Columns with missing values:')
display(missing_df)

fig, ax = plt.subplots(figsize=(7, 3))
missing_df['pct'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Missing Values (% of rows)')
ax.set_ylabel('Percentage (%)')
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Target Distribution — Class Imbalance

In [ ]:
counts = df['loan_status'].value_counts()
pcts   = df['loan_status'].value_counts(normalize=True) * 100
print('loan_status distribution:')
print(pd.DataFrame({'count': counts, 'pct': pcts.round(2)}))
imbalance_ratio = counts[0] / counts[1]
print(f'\nImbalance ratio (good:default): {imbalance_ratio:.2f}:1')

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Good (0)', 'Default (1)'], counts.values,
       color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.2)
for i, (v, p) in enumerate(zip(counts.values, pcts.values)):
    ax.text(i, v + 200, f'{v:,}\n({p:.1f}%)', ha='center', fontsize=10)
ax.set_title('Target Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Numeric Feature Distributions — Histogram + Boxplot

In [ ]:
numeric_cols = ['person_age', 'person_income', 'person_emp_length',
                'loan_amnt', 'loan_int_rate', 'loan_percent_income',
                'cb_person_cred_hist_length']

fig, axes = plt.subplots(len(numeric_cols), 2, figsize=(14, 4 * len(numeric_cols)))

for i, col in enumerate(numeric_cols):
    # Histogram colored by target
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        axes[i, 0].hist(df[df['loan_status'] == label][col].dropna(),
                        bins=40, alpha=0.6, color=color,
                        label=f'status={label}', density=True)
    axes[i, 0].set_title(f'{col} — histogram')
    axes[i, 0].legend()

    # Boxplot by target
    data = [df[df['loan_status'] == 0][col].dropna(),
            df[df['loan_status'] == 1][col].dropna()]
    bp = axes[i, 1].boxplot(data, patch_artist=True,
                             labels=['Good (0)', 'Default (1)'])
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')
    axes[i, 1].set_title(f'{col} — boxplot by status')

plt.suptitle('Numeric Feature Distributions', fontsize=15, y=1.002)
plt.tight_layout()
plt.show()

## 5. Categorical Feature Distributions — Bar Charts

In [ ]:
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = df.groupby([col, 'loan_status']).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'],
                edgecolor='white', linewidth=0.8)
    axes[i].set_title(f'{col} — default rate by category')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].set_xlabel('')
    axes[i].legend(['Good (0)', 'Default (1)'], loc='upper right')
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Categorical Feature Distributions vs Target', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap (Numeric Features)

In [ ]:
corr_cols = numeric_cols + ['loan_status']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Pearson Correlation — Numeric Features + Target', pad=12)
plt.tight_layout()
plt.show()

## 7. Scatter Plot — loan_amnt vs person_income by loan_status

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for label, color, name in [(0, '#2ecc71', 'Good (0)'), (1, '#e74c3c', 'Default (1)')]:
    sub = df[df['loan_status'] == label]
    ax.scatter(sub['person_income'], sub['loan_amnt'],
               c=color, alpha=0.35, s=12, label=name)

ax.set_xlabel('Person Annual Income')
ax.set_ylabel('Loan Amount')
ax.set_title('Loan Amount vs Annual Income (colored by loan_status)')
ax.legend()
# Cap x-axis for readability (long tail income outliers)
ax.set_xlim(0, df['person_income'].quantile(0.99))
plt.tight_layout()
plt.show()

## 8. Outlier Detection

In [ ]:
age_outliers = df[df['person_age'] > 100]
emp_outliers = df[df['person_emp_length'] > 60]

print(f'person_age > 100   : {len(age_outliers)} rows ({len(age_outliers)/len(df)*100:.2f}%)')
print(f'person_emp_length > 60 : {len(emp_outliers)} rows ({len(emp_outliers)/len(df)*100:.2f}%)')

print('\nAge outlier sample:')
display(age_outliers[['person_age', 'person_emp_length', 'person_income', 'loan_status']].head(10))

print('\nEmp-length outlier sample:')
display(emp_outliers[['person_age', 'person_emp_length', 'person_income', 'loan_status']].head(10))

# Visualise
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(df['person_age'], bins=60, color='steelblue', edgecolor='white')
ax1.axvline(100, color='red', linestyle='--', label='cutoff=100')
ax1.set_title('person_age distribution')
ax1.legend()

ax2.hist(df['person_emp_length'].dropna(), bins=60, color='steelblue', edgecolor='white')
ax2.axvline(60, color='red', linestyle='--', label='cutoff=60')
ax2.set_title('person_emp_length distribution')
ax2.legend()
plt.tight_layout()
plt.show()

## 9. Interest Rate & Default by Loan Grade

In [ ]:
grade_stats = df.groupby('loan_grade').agg(
    mean_int_rate=('loan_int_rate', 'mean'),
    default_rate=('loan_status', 'mean'),
    count=('loan_status', 'size')
).reset_index()

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
x = range(len(grade_stats))
ax1.bar(x, grade_stats['mean_int_rate'], color='#3498db', alpha=0.7, label='Mean Interest Rate')
ax2.plot(x, grade_stats['default_rate'] * 100, 'r-o', linewidth=2, label='Default Rate (%)')
ax1.set_xticks(x)
ax1.set_xticklabels(grade_stats['loan_grade'])
ax1.set_xlabel('Loan Grade')
ax1.set_ylabel('Mean Interest Rate (%)', color='#3498db')
ax2.set_ylabel('Default Rate (%)', color='red')
ax1.set_title('Mean Interest Rate & Default Rate by Loan Grade')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()
display(grade_stats)

## 10. Insight Summary

Berikut **5+ temuan penting** dari EDA:

### 1. Class Imbalance Signifikan
Dataset sangat imbalanced: ~78% pinjaman tidak default (status=0) vs ~22% default (status=1). Rasio sekitar 3.5:1 ini berarti model naive yang selalu prediksi "good" pun bisa punya akurasi 78%. Perlu SMOTE atau class weighting saat training.

### 2. Outlier Tidak Realistis di person_age dan person_emp_length
Terdapat nilai `person_age` hingga **144 tahun** dan `person_emp_length` hingga **123 tahun** — keduanya tidak mungkin secara biologis. Baris-baris ini adalah data entry error dan harus dihapus (age > 100, emp_length > 60) sebelum training.

### 3. Loan Grade adalah Prediktor Terkuat
Default rate meningkat monoton dari Grade A (~5%) hingga Grade G (~40%). Grade mencerminkan risk assessment bank itu sendiri, menjadikannya fitur paling informatif. Interest rate pun berkorelasi positif dengan grade, yang wajar.

### 4. loan_percent_income Berkorelasi Positif dengan Default
Peminjam yang mengalokasikan proporsi pendapatan lebih besar untuk cicilan pinjaman cenderung lebih sering default. Fitur ini dan `loan_int_rate` memiliki korelasi positif paling tinggi terhadap `loan_status` di antara fitur numerik.

### 5. Riwayat Default (cb_person_default_on_file) Sangat Prediktif
Peminjam dengan riwayat default sebelumnya (`Y`) memiliki default rate jauh lebih tinggi dibanding yang bersih (`N`). Ini adalah sinyal historical risk yang kuat dan harus di-encode dengan benar (bukan one-hot).

### 6. Missing Values Terkonsentrasi di 2 Kolom
- `loan_int_rate`: 3.116 null (~9.5%) — kemungkinan pending approval atau data tidak tersimpan
- `person_emp_length`: 895 null (~2.7%) — peminjam mungkin tidak bekerja (pengangguran/pelajar)  
Keduanya di-impute dengan **median per loan_grade** agar tetap mempertahankan distribusi risk tier.

### 7. Income Distribution Sangat Right-Skewed
Sebagian besar peminjam berpendapatan di bawah \$100K/tahun, tetapi ada ekor panjang hingga \$6M+. Normalisasi StandardScaler (atau RobustScaler) diperlukan untuk model sensitif terhadap skala (Logistic Regression, SVM, dll).
